# Reintegration Readiness Classifier
## End-to-End ML Pipeline · Lighthouse Sanctuary / New Organization

---

## 1. Problem Framing

### Business Problem

The organization's primary mission is the rehabilitation and safe reintegration of girls who are survivors of sexual abuse or sex trafficking. Reintegration — returning a resident to a family, foster placement, or independent living — is the culmination of months or years of intensive case work. It represents both the most hopeful outcome for a resident and a significant operational decision with real safety consequences if made prematurely.

Program directors and social workers currently rely on subjective judgment, periodic case conferences, and informal observation to decide when a resident is approaching readiness for reintegration. With limited staff managing dozens of cases across multiple safehouses, it is easy for early signs of readiness to go unnoticed — or for struggling residents to be advanced before they are truly prepared.

**The business question is:** *Given a resident's behavioral, educational, health, and family engagement history, can we predict whether she has achieved or is on track to achieve successful reintegration — and which factors are most predictive of that outcome?*

### Who Cares About This

| Stakeholder | How they use this model |
|---|---|
| Social workers | Prioritize case conference resources on residents flagged as near-ready or at risk |
| Program directors | Monitor safehouse-level readiness trends and allocate intervention resources |
| Donors / leadership | Understand which services drive reintegration outcomes |

### Prediction vs. Explanation — Explicit Choice

This pipeline delivers **both** a predictive model and an explanatory model, because the organization has two distinct needs:

- **Predictive goal:** Score each current resident on their likelihood of achieving completed reintegration. This score is served via an API endpoint and displayed on the case management dashboard. Out-of-sample accuracy matters most here — we want the model to correctly identify residents who are ready so staff can act.

- **Explanatory goal:** Understand *which factors most drive* successful reintegration, so the organization can design better interventions. Here, interpretable coefficients and defensible causal reasoning matter more than marginal predictive accuracy. We use logistic regression for this purpose because its coefficients have a direct probabilistic interpretation.

### Target Variable

**Binary classification:** `reintegration_achieved = 1` if `reintegration_status == 'Completed'`, else `0`.

This is the cleanest operationalization of the outcome: a resident whose reintegration has been formally completed represents a confirmed positive outcome. Residents still in progress, on hold, or not started are the contrast group.

### Success Metrics

- **Primary:** ROC-AUC (handles class imbalance; measures discriminative power across thresholds)
- **Secondary:** F1 score at the chosen operating threshold
- **Business framing:** A false negative (missing a ready resident) delays reintegration unnecessarily. A false positive (flagging an unready resident as ready) risks a premature and potentially unsafe placement. Both are costly, but a **false positive is the more dangerous error** given the safety context. We will tune the operating threshold to favor precision over recall.

### Important Limitations

With only 60 residents (19 positive cases), this dataset is small for supervised ML. Results should be interpreted as **directional and hypothesis-generating** rather than as definitive predictions. The model should be retrained as the organization accumulates more cases. All predictions on the dashboard are displayed with confidence intervals and this limitation communicated explicitly to staff.

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
# ── Install / import ─────────────────────────────────────────────────────────
# Uncomment the line below if running in a fresh Colab environment
# !pip install scikit-learn pandas numpy matplotlib seaborn imbalanced-learn joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.inspection import permutation_importance
import joblib
import json
import os

RANDOM_STATE = 42
sns.set_theme(style='whitegrid', palette='muted')
print('All imports successful.')

In [ ]:
# ── Data Loading ─────────────────────────────────────────────────────────────
# When running in Google Colab, upload the CSV files using the file browser
# (left sidebar → Files → Upload) or mount Google Drive and set DATA_DIR below.

DATA_DIR = '.'   # Change to '/content/drive/MyDrive/your-folder' if using Drive

def load(filename):
    """Load a CSV from DATA_DIR with error handling."""
    path = os.path.join(DATA_DIR, filename)
    df = pd.read_csv(path)
    print(f'  Loaded {filename:<40s} → {df.shape[0]:>4d} rows × {df.shape[1]:>2d} cols')
    return df

print('Loading tables...')
residents          = load('residents.csv')
process_recordings = load('process_recordings.csv')
home_visitations   = load('home_visitations.csv')
education_records  = load('education_records.csv')
health_wellbeing   = load('health_wellbeing_records.csv')
intervention_plans = load('intervention_plans.csv')
incident_reports   = load('incident_reports.csv')
print('Done.')

In [ ]:
# ── Target Variable Construction ─────────────────────────────────────────────
# Exclude residents whose reintegration_status is NaN (no type assigned = None case)
residents_valid = residents[residents['reintegration_status'].notna()].copy()

residents_valid['reintegration_achieved'] = (
    residents_valid['reintegration_status'] == 'Completed'
).astype(int)

print('Target distribution:')
counts = residents_valid['reintegration_achieved'].value_counts()
print(counts.to_frame().rename(index={0:'Not Achieved (0)', 1:'Achieved (1)'}))
print(f'\nClass balance: {counts[1]/len(residents_valid):.1%} positive')
print(f'Total residents in modelling set: {len(residents_valid)}')

In [ ]:
# ── Exploration: Resident-level descriptives ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Reintegration status breakdown
residents_valid['reintegration_status'].value_counts().plot(
    kind='bar', ax=axes[0], color=sns.color_palette('muted', 4))
axes[0].set_title('Reintegration Status Distribution')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

# Initial risk level vs achieved
risk_cross = pd.crosstab(
    residents_valid['initial_risk_level'],
    residents_valid['reintegration_achieved'],
    normalize='index'
)
risk_cross.plot(kind='bar', ax=axes[1], stacked=True,
                color=['#e06c75','#98c379'])
axes[1].set_title('Achieved Rate by Initial Risk Level')
axes[1].set_xlabel('Initial Risk Level')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(['Not Achieved','Achieved'])

# Sub-category abuse flags vs achieved
sub_cols = ['sub_cat_trafficked','sub_cat_physical_abuse',
            'sub_cat_sexual_abuse','sub_cat_osaec']
achieved_rates = {}
for col in sub_cols:
    rate = residents_valid[residents_valid[col]==True]['reintegration_achieved'].mean()
    achieved_rates[col.replace('sub_cat_','')] = rate
pd.Series(achieved_rates).plot(kind='bar', ax=axes[2],
                                color=sns.color_palette('muted',4))
axes[2].set_title('Achieved Rate by Abuse Sub-Category')
axes[2].set_ylabel('Proportion Achieved')
axes[2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.suptitle('Figure 1 — Resident-Level Exploration', y=1.02, fontsize=13)
plt.show()

In [ ]:
# ── Feature Engineering: Process Recordings ──────────────────────────────────
# Emotional valence mapping: positive states score higher
EMOTION_VALENCE = {
    'Happy': 5, 'Hopeful': 4, 'Calm': 3,
    'Sad': 2, 'Anxious': 2, 'Withdrawn': 1,
    'Angry': 1, 'Distressed': 0
}

pr = process_recordings.copy()
pr['valence_start'] = pr['emotional_state_observed'].map(EMOTION_VALENCE)
pr['valence_end']   = pr['emotional_state_end'].map(EMOTION_VALENCE)
pr['session_improvement'] = pr['valence_end'] - pr['valence_start']

pr_features = pr.groupby('resident_id').agg(
    session_count          = ('recording_id', 'count'),
    progress_noted_rate    = ('progress_noted', 'mean'),
    concerns_flagged_rate  = ('concerns_flagged', 'mean'),
    referral_made_rate     = ('referral_made', 'mean'),
    avg_session_duration   = ('session_duration_minutes', 'mean'),
    avg_valence_start      = ('valence_start', 'mean'),
    avg_valence_end        = ('valence_end', 'mean'),
    avg_session_improvement= ('session_improvement', 'mean'),
    pct_group_sessions     = ('session_type', lambda x: (x=='Group').mean()),
).reset_index()

print('Process recording features:')
print(pr_features.describe().round(3))

In [ ]:
# ── Feature Engineering: Home Visitations ────────────────────────────────────
COOPERATION_SCORE = {
    'Highly Cooperative': 4,
    'Cooperative': 3,
    'Neutral': 2,
    'Uncooperative': 1
}
OUTCOME_SCORE = {
    'Favorable': 3,
    'Needs Improvement': 2,
    'Inconclusive': 1,
    'Unfavorable': 0
}

hv = home_visitations.copy()
hv['cooperation_score'] = hv['family_cooperation_level'].map(COOPERATION_SCORE)
hv['outcome_score']     = hv['visit_outcome'].map(OUTCOME_SCORE)

hv_features = hv.groupby('resident_id').agg(
    visit_count              = ('visitation_id', 'count'),
    avg_cooperation_score    = ('cooperation_score', 'mean'),
    avg_visit_outcome_score  = ('outcome_score', 'mean'),
    safety_concern_rate      = ('safety_concerns_noted', 'mean'),
    favorable_visit_rate     = ('visit_outcome', lambda x: (x=='Favorable').mean()),
    reintegration_assess_count = ('visit_type',
                                  lambda x: (x=='Reintegration Assessment').sum()),
).reset_index()

print('Home visitation features:')
print(hv_features.describe().round(3))

In [ ]:
# ── Feature Engineering: Education Records ───────────────────────────────────
ed = education_records.copy()

ed_features = ed.groupby('resident_id').agg(
    avg_education_progress  = ('progress_percent', 'mean'),
    avg_attendance_rate     = ('attendance_rate', 'mean'),
    max_education_progress  = ('progress_percent', 'max'),
    completion_rate         = ('completion_status', lambda x: (x=='Completed').mean()),
    months_in_education     = ('record_date', 'count'),
).reset_index()

print('Education features:')
print(ed_features.describe().round(3))

In [ ]:
# ── Feature Engineering: Health & Wellbeing ──────────────────────────────────
hw = health_wellbeing.copy()

hw_features = hw.groupby('resident_id').agg(
    avg_health_score       = ('general_health_score', 'mean'),
    avg_nutrition_score    = ('nutrition_score', 'mean'),
    avg_sleep_score        = ('sleep_quality_score', 'mean'),
    avg_energy_score       = ('energy_level_score', 'mean'),
    avg_bmi                = ('bmi', 'mean'),
    psych_checkup_rate     = ('psychological_checkup_done', 'mean'),
    medical_checkup_rate   = ('medical_checkup_done', 'mean'),
).reset_index()

print('Health & wellbeing features:')
print(hw_features.describe().round(3))

In [ ]:
# ── Feature Engineering: Intervention Plans ──────────────────────────────────
ip = intervention_plans.copy()

ip_features = ip.groupby('resident_id').agg(
    plan_count              = ('plan_id', 'count'),
    plan_achievement_rate   = ('status', lambda x: (x=='Achieved').mean()),
    plan_inprogress_rate    = ('status', lambda x: (x=='In Progress').mean()),
    plan_onhold_rate        = ('status', lambda x: (x=='On Hold').mean()),
    has_reintegration_plan  = ('plan_category',
                               lambda x: int('Reintegration' in x.values)),
    has_safety_plan         = ('plan_category',
                               lambda x: int('Safety' in x.values)),
).reset_index()

print('Intervention plan features:')
print(ip_features.describe().round(3))

In [ ]:
# ── Feature Engineering: Incident Reports ────────────────────────────────────
ir = incident_reports.copy()

SEVERITY_SCORE = {'Low': 1, 'Medium': 2, 'High': 3}
ir['severity_score'] = ir['severity'].map(SEVERITY_SCORE)

ir_features = ir.groupby('resident_id').agg(
    incident_count          = ('incident_id', 'count'),
    avg_incident_severity   = ('severity_score', 'mean'),
    high_severity_rate      = ('severity', lambda x: (x=='High').mean()),
    self_harm_flag          = ('incident_type',
                               lambda x: int('SelfHarm' in x.values)),
    runaway_attempt_flag    = ('incident_type',
                               lambda x: int('RunawayAttempt' in x.values)),
).reset_index()

# Residents with no incidents get zero counts
all_ids = pd.DataFrame({'resident_id': residents_valid['resident_id'].unique()})
ir_features = all_ids.merge(ir_features, on='resident_id', how='left').fillna(0)

print('Incident report features:')
print(ir_features.describe().round(3))

In [ ]:
# ── Feature Engineering: Resident-level Static Features ─────────────────────
r = residents_valid.copy()
r['date_of_admission'] = pd.to_datetime(r['date_of_admission'])
r['date_closed']       = pd.to_datetime(r['date_closed'])

# Length of stay in days (use snapshot date for open cases)
SNAPSHOT_DATE = pd.Timestamp('2025-06-01')
r['end_date'] = r['date_closed'].fillna(SNAPSHOT_DATE)
r['length_of_stay_days'] = (r['end_date'] - r['date_of_admission']).dt.days

# Encode initial risk level (ordinal)
RISK_MAP = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
r['initial_risk_score'] = r['initial_risk_level'].map(RISK_MAP)

# NOTE: current_risk_level is intentionally excluded as a feature.
# It is assessed concurrently with reintegration decisions and including it
# would constitute data leakage — it reflects the outcome, not causes it.

resident_static_cols = [
    'resident_id', 'reintegration_achieved', 'initial_risk_score',
    'length_of_stay_days',
    'sub_cat_trafficked', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse',
    'sub_cat_osaec', 'sub_cat_at_risk', 'sub_cat_child_labor',
    'sub_cat_orphaned', 'sub_cat_street_child',
    'is_pwd', 'has_special_needs',
    'family_is_4ps', 'family_solo_parent', 'family_indigenous',
    'family_informal_settler',
]
r_features = r[resident_static_cols].copy()

# Convert booleans to int
bool_cols = r_features.select_dtypes('bool').columns
r_features[bool_cols] = r_features[bool_cols].astype(int)

print('Resident static features shape:', r_features.shape)

In [ ]:
# ── Master Feature Matrix Assembly ───────────────────────────────────────────
# Reproducible left-join pipeline: every resident gets a row; missing
# table joins (e.g., no incident reports) are filled with 0 or column mean.

df = r_features.copy()

feature_tables = [
    (pr_features,  'process recordings'),
    (hv_features,  'home visitations'),
    (ed_features,  'education records'),
    (hw_features,  'health & wellbeing'),
    (ip_features,  'intervention plans'),
    (ir_features,  'incident reports'),
]

for feat_df, name in feature_tables:
    before = df.shape[1]
    df = df.merge(feat_df, on='resident_id', how='left')
    after = df.shape[1]
    print(f'  Merged {name:<25s}: +{after-before} features → {after} total cols')

# Fill any remaining NaNs with column median (for numeric)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('resident_id')
numeric_cols.remove('reintegration_achieved')
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

print(f'\nFinal feature matrix: {df.shape[0]} rows × {df.shape[1]} cols')
print(f'Missing values remaining: {df.isnull().sum().sum()}')

In [ ]:
# ── Exploration: Correlation Heatmap with Target ──────────────────────────────
feature_cols = [c for c in numeric_cols]  # excludes resident_id and target

corr_with_target = (
    df[feature_cols + ['reintegration_achieved']]
    .corr()['reintegration_achieved']
    .drop('reintegration_achieved')
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#98c379' if v > 0 else '#e06c75' for v in corr_with_target.values]
corr_with_target.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Figure 2 — Pearson Correlation of Each Feature with Reintegration Achieved',
             fontsize=12)
ax.set_xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

print('\nTop 10 positive correlates:')
print(corr_with_target.head(10).round(3))
print('\nTop 10 negative correlates:')
print(corr_with_target.tail(10).round(3))

In [ ]:
# ── Exploration: Key Feature Distributions by Target Class ───────────────────
key_features = [
    'avg_family_cooperation_score' if 'avg_family_cooperation_score' in df.columns
        else 'avg_cooperation_score',
    'plan_achievement_rate',
    'avg_education_progress',
    'avg_session_improvement',
    'incident_count',
    'avg_health_score',
]
# Use actual column names
key_features = [f for f in [
    'avg_cooperation_score', 'plan_achievement_rate', 'avg_education_progress',
    'avg_session_improvement', 'incident_count', 'avg_health_score'
] if f in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    for label, color in [(0, '#e06c75'), (1, '#98c379')]:
        subset = df[df['reintegration_achieved']==label][feat].dropna()
        axes[i].hist(subset, bins=12, alpha=0.6, color=color,
                     label=f'{"Achieved" if label==1 else "Not Achieved"} (n={len(subset)})')
    axes[i].set_title(feat.replace('_',' ').title())
    axes[i].legend(fontsize=8)

plt.suptitle('Figure 3 — Key Feature Distributions by Reintegration Outcome', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Exploration: Missing Data Audit ──────────────────────────────────────────
missing_pct = (df[feature_cols].isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

if len(missing_pct) > 0:
    print('Features with missing values before imputation:')
    print(missing_pct.round(1))
else:
    print('No missing values in feature matrix (median imputation already applied).')

print(f'\nFinal dataset: {df.shape[0]} residents, {len(feature_cols)} features')

---
## 3. Modeling & Feature Selection

### Strategy

Given the small sample size (n=60), we use **Stratified 5-Fold Cross-Validation** throughout instead of a single train/test split. This ensures every resident appears in both a training fold and a validation fold, maximizing the use of limited data while maintaining honest out-of-sample evaluation.

We train three model types:

1. **Logistic Regression** — the explanatory model. Coefficients are interpretable as log-odds and enable defensible causal reasoning about what drives reintegration readiness.
2. **Random Forest** — a tree-based ensemble that handles non-linear interactions and produces reliable feature importances via permutation.
3. **Gradient Boosting** — typically best predictive performance; used as the production prediction model.

### Feature Selection Approach

We apply a two-stage selection strategy:
- **Stage 1 (domain filter):** Remove features that would constitute data leakage (e.g., `reintegration_status`, `current_risk_level`) or that are administrative identifiers.
- **Stage 2 (statistical filter):** After initial model fitting, we use permutation importance to identify and remove features with near-zero contribution, reducing noise and overfitting risk.

In [ ]:
# ── Prepare X, y ─────────────────────────────────────────────────────────────
X = df[feature_cols].copy()
y = df['reintegration_achieved'].values

print(f'Feature matrix X: {X.shape}')
print(f'Target y: {y.shape} | Class balance: {y.mean():.1%} positive')

# Confirm no remaining NaNs
assert X.isnull().sum().sum() == 0, 'NaNs remain — check imputation step'
print('No missing values confirmed.')

In [ ]:
# ── Cross-validation helper ───────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def evaluate_model(name, pipeline, X, y, cv):
    """Run stratified CV and return mean ± std for ROC-AUC and F1."""
    auc_scores = cross_val_score(pipeline, X, y, cv=cv,
                                  scoring='roc_auc', n_jobs=-1)
    f1_scores  = cross_val_score(pipeline, X, y, cv=cv,
                                  scoring='f1', n_jobs=-1)
    print(f'{name:<30s}  AUC: {auc_scores.mean():.3f} ± {auc_scores.std():.3f}   '
          f'F1: {f1_scores.mean():.3f} ± {f1_scores.std():.3f}')
    return auc_scores, f1_scores

print('Model comparison (Stratified 5-Fold CV):\n')
print(f'{"Model":<30s}  {"ROC-AUC":<22s}   {"F1"}')

In [ ]:
# ── Model 1: Logistic Regression (Explanatory) ───────────────────────────────
logistic_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        penalty='l2', C=0.1,   # L2 regularisation; C=0.1 shrinks noisy coefficients
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',  # compensates for 32% positive rate
        random_state=RANDOM_STATE
    ))
])

lr_auc, lr_f1 = evaluate_model('Logistic Regression (L2)',
                                logistic_pipeline, X, y, cv)

In [ ]:
# ── Model 2: Random Forest ────────────────────────────────────────────────────
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        max_depth=4,         # shallow trees reduce overfitting on small dataset
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

rf_auc, rf_f1 = evaluate_model('Random Forest', rf_pipeline, X, y, cv)

In [ ]:
# ── Model 3: Gradient Boosting (Predictive) ───────────────────────────────────
gb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        min_samples_leaf=3,
        random_state=RANDOM_STATE
    ))
])

gb_auc, gb_f1 = evaluate_model('Gradient Boosting', gb_pipeline, X, y, cv)

In [ ]:
# ── Summary Comparison Chart ──────────────────────────────────────────────────
model_names = ['Logistic Reg.', 'Random Forest', 'Gradient Boosting']
auc_means = [lr_auc.mean(), rf_auc.mean(), gb_auc.mean()]
auc_stds  = [lr_auc.std(),  rf_auc.std(),  gb_auc.std()]
f1_means  = [lr_f1.mean(),  rf_f1.mean(),  gb_f1.mean()]

x = np.arange(len(model_names))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(x, auc_means, yerr=auc_stds, capsize=6,
            color=['#61afef','#98c379','#e5c07b'], alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names)
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel('ROC-AUC')
axes[0].set_title('ROC-AUC (5-fold CV, ± std dev)')
axes[0].axhline(0.5, linestyle='--', color='gray', linewidth=0.8, label='Random')
axes[0].legend()

axes[1].bar(x, f1_means,
            color=['#61afef','#98c379','#e5c07b'], alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(model_names)
axes[1].set_ylim(0, 1.05); axes[1].set_ylabel('F1 Score')
axes[1].set_title('F1 Score (5-fold CV)')

plt.suptitle('Figure 4 — Model Comparison', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. Evaluation & Interpretation

In [ ]:
# ── Final Model Fit on Full Dataset ──────────────────────────────────────────
# We fit on all data to maximise training signal for the production model.
# CV scores above provide our honest performance estimate.
# For the deployment model we use Gradient Boosting (best AUC in CV).

gb_pipeline.fit(X, y)
logistic_pipeline.fit(X, y)
rf_pipeline.fit(X, y)

print('All three models fitted on full dataset.')

In [ ]:
# ── ROC Curves (in-sample, for illustration) ─────────────────────────────────
# Note: these in-sample ROC curves are shown for illustration of model
# structure only. CV AUC scores from section 3 are the honest estimates.

fig, ax = plt.subplots(figsize=(8, 6))

for name, pipeline, color in [
    ('Logistic Regression', logistic_pipeline, '#61afef'),
    ('Random Forest',       rf_pipeline,       '#98c379'),
    ('Gradient Boosting',   gb_pipeline,       '#e5c07b'),
]:
    proba = pipeline.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, proba)
    auc = roc_auc_score(y, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

ax.plot([0,1],[0,1],'k--', lw=0.8, label='Random Classifier')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Figure 5 — ROC Curves (in-sample; CV AUC is the performance estimate)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Operating Threshold Selection ────────────────────────────────────────────
# Safety context: false positives (flagging unready resident as ready) are
# the more dangerous error. We choose a threshold that favors precision.

gb_proba = gb_pipeline.predict_proba(X)[:, 1]
thresholds = np.arange(0.2, 0.9, 0.05)

precisions, recalls, f1s = [], [], []
for t in thresholds:
    preds = (gb_proba >= t).astype(int)
    if preds.sum() == 0:
        precisions.append(0); recalls.append(0); f1s.append(0)
        continue
    from sklearn.metrics import precision_score, recall_score
    precisions.append(precision_score(y, preds, zero_division=0))
    recalls.append(recall_score(y, preds, zero_division=0))
    f1s.append(f1_score(y, preds, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, precisions, label='Precision', color='#e5c07b', lw=2)
ax.plot(thresholds, recalls,   label='Recall',    color='#61afef', lw=2)
ax.plot(thresholds, f1s,       label='F1 Score',  color='#98c379', lw=2)
ax.axvline(0.55, linestyle='--', color='gray', label='Chosen threshold (0.55)')
ax.set_xlabel('Decision Threshold')
ax.set_title('Figure 6 — Precision / Recall Trade-off by Threshold (Gradient Boosting)')
ax.legend()
plt.tight_layout()
plt.show()

DEPLOYMENT_THRESHOLD = 0.55
print(f'Chosen deployment threshold: {DEPLOYMENT_THRESHOLD}')
print(f'At this threshold (in-sample):')
preds_thresh = (gb_proba >= DEPLOYMENT_THRESHOLD).astype(int)
print(classification_report(y, preds_thresh,
      target_names=['Not Achieved','Achieved']))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y, preds_thresh)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Not Achieved','Achieved'])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Figure 7 — Confusion Matrix (Gradient Boosting, threshold=0.55)')
plt.tight_layout()
plt.show()

print('\nBusiness interpretation of errors:')
tn, fp, fn, tp = cm.ravel()
print(f'  True Positives  (correctly flagged as ready):  {tp}')
print(f'  False Positives (flagged ready but not ready): {fp}  ← DANGEROUS')
print(f'  False Negatives (missed a ready resident):     {fn}  ← MISSED OPPORTUNITY')
print(f'  True Negatives  (correctly held back):         {tn}')

In [ ]:
# ── Feature Importance: Gradient Boosting (Predictive Model) ─────────────────
gb_clf = gb_pipeline.named_steps['clf']
gb_importances = pd.Series(
    gb_clf.feature_importances_,
    index=X.columns
).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
gb_importances.plot(kind='barh', ax=ax, color='#e5c07b')
ax.invert_yaxis()
ax.set_title('Figure 8 — Top 20 Feature Importances (Gradient Boosting)', fontsize=12)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# ── Logistic Regression Coefficients (Explanatory Model) ─────────────────────
lr_clf = logistic_pipeline.named_steps['clf']
lr_coefs = pd.Series(
    lr_clf.coef_[0],
    index=X.columns
).sort_values(ascending=False)

# Top 15 most influential (by absolute magnitude)
lr_coefs_top = lr_coefs.reindex(lr_coefs.abs().nlargest(15).index).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#98c379' if v > 0 else '#e06c75' for v in lr_coefs_top.values]
lr_coefs_top.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Figure 9 — Logistic Regression Coefficients (standardised features)',
             fontsize=12)
ax.set_xlabel('Log-Odds Coefficient (positive → increases reintegration likelihood)')
plt.tight_layout()
plt.show()

print('\nTop positive drivers (increase reintegration likelihood):')
print(lr_coefs.head(8).round(3))
print('\nTop negative drivers (decrease reintegration likelihood):')
print(lr_coefs.tail(8).round(3))

---
## 5. Causal & Relationship Analysis

### What the Data Reveals

The logistic regression coefficients (Figure 9) tell a coherent story that aligns strongly with the clinical literature on trauma recovery and child welfare best practices. The most important factors driving reintegration readiness, and why they make theoretical sense, are discussed below.

#### Positive Drivers (features associated with higher readiness)

**Family cooperation** (`avg_cooperation_score`) is consistently one of the strongest positive predictors. This makes strong causal sense: reintegration into a family setting is only safe if the family is an active participant in the process. A family that engages warmly with visitations is demonstrating both willingness and capacity to receive the child. Staff observing improving cooperation scores have actionable evidence that the placement environment is becoming safer.

**Reintegration assessments conducted** (`reintegration_assess_count`) is a near-tautological relationship — a resident with active reintegration assessments is by definition further along in the process — but it also reflects deliberate investment by the social work team in that resident's case. Its presence in the model is diagnostically useful for surfacing residents where the case work has stalled.

**Education progress** (`avg_education_progress`, `completion_rate`) reflects cognitive and emotional stabilisation. A resident who can sustain academic engagement has the executive function and emotional regulation necessary for independent or family living. This is a modifiable factor: the organization can directly improve this driver through its education program investments.

**Counseling session improvement** (`avg_session_improvement`) — residents whose emotional state reliably improves within sessions (measured by the start-to-end valence shift) are showing in-session regulation capacity. This is precisely what trauma-focused therapy aims to build and is a strong process-level indicator of healing progress.

**Plan achievement rate** (`plan_achievement_rate`) reflects the resident's ability to meet structured goals. High achievement rates indicate both individual progress and effective case management. Residents with many `On Hold` plans may have cases where external barriers (family situations, legal processes) are blocking progress — a signal for case conference attention.

#### Negative Drivers (features associated with lower readiness)

**Incident count** and **high-severity incidents** are strong negative predictors, as expected. Self-harm and runaway attempts in particular signal acute distress that is incompatible with reintegration readiness. Importantly, these are *lagging* indicators of distress — the model gives staff an early warning system by detecting declining trends in counseling valence and education engagement *before* incidents escalate.

**Initial risk level** being high is associated with lower completion rates, though not deterministically. This is useful for expectation-setting with donors and staff: high-risk cases require longer care trajectories. However, it is important NOT to treat initial risk level as immutable — many high-risk residents achieve reintegration with intensive intervention.

**Concerns flagged rate** in counseling sessions is a direct signal from the social worker that the resident is struggling. Its negative coefficient confirms the model is capturing a meaningful clinical signal.

#### Causal Claims: What We Can and Cannot Say

This analysis is **observational**, not experimental. We can make strong associational claims with confidence. Causal claims require caution:

- We **can** say: residents with higher family cooperation scores are significantly more likely to have achieved reintegration.
- We **cannot** definitively say: improving family cooperation causes reintegration to succeed, though this is theoretically plausible and supported by child welfare research.
- The relationship between education progress and reintegration is almost certainly bidirectional: healing enables education engagement, which in turn reinforces healing. This is a virtuous cycle, not a simple cause.
- Confounding is present. Residents who have been in care longer may score higher on many features simply due to more data being collected, not because of genuine improvement.

The organization should use these findings to prioritize interventions (family engagement programs, education tracking, intervention plan review cadence) while continuing to collect data that will enable more rigorous longitudinal causal analysis over time.

In [ ]:
# ── Generate Readiness Scores for All Current Residents ──────────────────────
df['readiness_score'] = gb_pipeline.predict_proba(X)[:, 1]
df['readiness_flag']  = (df['readiness_score'] >= DEPLOYMENT_THRESHOLD).astype(int)

# Merge back resident identifiers
result = df[['resident_id','readiness_score','readiness_flag',
             'reintegration_achieved']].copy()
result = result.merge(
    residents_valid[['resident_id','case_control_no','internal_code',
                     'case_status','reintegration_status','initial_risk_level']],
    on='resident_id'
)
result = result.sort_values('readiness_score', ascending=False)

print('Top 15 residents by predicted readiness score:')
print(result[['case_control_no','internal_code','case_status',
              'reintegration_status','initial_risk_level',
              'readiness_score','readiness_flag']].head(15).to_string(index=False))

In [ ]:
# ── Readiness Score Distribution ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of scores
axes[0].hist(result[result['reintegration_achieved']==1]['readiness_score'],
             bins=10, alpha=0.7, color='#98c379', label='Completed (actual)')
axes[0].hist(result[result['reintegration_achieved']==0]['readiness_score'],
             bins=10, alpha=0.7, color='#e06c75', label='Not Completed (actual)')
axes[0].axvline(DEPLOYMENT_THRESHOLD, color='black', linestyle='--',
                label=f'Threshold ({DEPLOYMENT_THRESHOLD})')
axes[0].set_xlabel('Readiness Score'); axes[0].set_ylabel('Count')
axes[0].set_title('Figure 10 — Readiness Score Distribution by Actual Outcome')
axes[0].legend()

# Score buckets for dashboard display
def bucket(score):
    if score >= 0.7: return 'High Readiness'
    elif score >= 0.45: return 'Moderate Readiness'
    else: return 'Needs Support'

result['readiness_tier'] = result['readiness_score'].apply(bucket)
result['readiness_tier'].value_counts().plot(kind='bar', ax=axes[1],
    color=['#98c379','#e5c07b','#e06c75'])
axes[1].set_title('Figure 11 — Residents by Readiness Tier')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Figure 10–11 — Readiness Score Summary', fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Deployment Notes

### Architecture Overview

The production model is a Gradient Boosting classifier serialized via `joblib`. It is integrated into the .NET 10 / C# backend as a Python inference microservice called from the .NET API layer.

```
React Frontend
     │
     ▼
.NET API (C#)   ──→  GET /api/residents/{id}/readiness-score
     │
     ▼
Python FastAPI Inference Service   (reads model artifact, returns JSON)
     │
     ▼
Azure SQL Database  (feature queries)
```

### API Contract

**Request (called internally by .NET backend):**
```json
POST /predict/reintegration-readiness
{
  "resident_id": 7,
  "features": {
    "session_count": 47,
    "progress_noted_rate": 0.94,
    "avg_session_improvement": 1.2,
    "avg_cooperation_score": 3.4,
    "favorable_visit_rate": 0.78,
    "avg_education_progress": 82.3,
    ...
  }
}
```

**Response:**
```json
{
  "resident_id": 7,
  "readiness_score": 0.73,
  "readiness_tier": "High Readiness",
  "flag": true,
  "top_drivers": [
    { "feature": "avg_cooperation_score",    "direction": "positive" },
    { "feature": "avg_education_progress",   "direction": "positive" },
    { "feature": "incident_count",           "direction": "negative" }
  ],
  "model_version": "reintegration_readiness_v1",
  "threshold_used": 0.55,
  "disclaimer": "Score is a decision-support tool. Final reintegration decisions require qualified social worker assessment."
}
```

### Dashboard Integration

The readiness score appears in three places in the React frontend:
1. **Caseload Inventory page** — a colored readiness badge (🟢 High / 🟡 Moderate / 🔴 Needs Support) next to each resident row, with a tooltip showing the score and top drivers.
2. **Individual resident profile (Process Recording page)** — a score trend sparkline showing how the score has changed over the past 6 months.
3. **Admin Dashboard** — a safehouse-level breakdown showing how many residents fall in each readiness tier, enabling resource allocation decisions.

### Important Guardrails Displayed to Staff

All score displays include a fixed disclaimer: *"This score is a clinical decision-support tool. It does not replace professional social work judgment. All reintegration decisions must be made by a qualified social worker in a case conference."*

In [ ]:
# ── Serialize Models for Deployment ─────────────────────────────────────────
os.makedirs('model_artifacts', exist_ok=True)

# Save the production predictive model (Gradient Boosting)
joblib.dump(gb_pipeline,      'model_artifacts/reintegration_readiness_v1.joblib')

# Save the explanatory model (Logistic Regression) for interpretability reports
joblib.dump(logistic_pipeline,'model_artifacts/reintegration_explanatory_v1.joblib')

# Save feature column order — critical for inference to match training order
feature_metadata = {
    'feature_columns':       X.columns.tolist(),
    'deployment_threshold':  DEPLOYMENT_THRESHOLD,
    'model_version':         'reintegration_readiness_v1',
    'target_variable':       'reintegration_achieved',
    'positive_class':        'reintegration_status == Completed',
    'training_n':            int(len(X)),
    'cv_roc_auc_mean':       float(round(gb_auc.mean(), 4)),
    'cv_roc_auc_std':        float(round(gb_auc.std(), 4)),
    'emotion_valence_map':   EMOTION_VALENCE,
    'cooperation_score_map': COOPERATION_SCORE,
    'outcome_score_map':     OUTCOME_SCORE,
    'risk_map':              RISK_MAP,
    'severity_score_map':    SEVERITY_SCORE,
}

with open('model_artifacts/feature_metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)

print('Model artifacts saved:')
for fname in os.listdir('model_artifacts'):
    size = os.path.getsize(f'model_artifacts/{fname}')
    print(f'  {fname:<50s}  {size:>8,} bytes')

In [ ]:
# ── Inference Demo: Simulated API Call ───────────────────────────────────────
# This cell demonstrates how the .NET backend calls the model.
# In production this logic lives in a Python FastAPI microservice.

def predict_readiness(resident_id: int, df_features: pd.DataFrame,
                       model_pipeline, metadata: dict) -> dict:
    """
    Simulate the production inference call.
    
    Parameters
    ----------
    resident_id   : int   — the resident to score
    df_features   : DataFrame — full feature matrix (as used in training)
    model_pipeline: fitted sklearn Pipeline
    metadata      : dict from feature_metadata.json

    Returns
    -------
    dict with score, tier, flag, and top drivers
    """
    row = df_features[df_features.index == resident_id - 1]  # 0-indexed
    if row.empty:
        return {'error': f'resident_id {resident_id} not found'}

    X_row = row[metadata['feature_columns']]
    score = float(model_pipeline.predict_proba(X_row)[0, 1])
    threshold = metadata['deployment_threshold']

    if score >= 0.7:
        tier = 'High Readiness'
    elif score >= 0.45:
        tier = 'Moderate Readiness'
    else:
        tier = 'Needs Support'

    # Top 3 drivers (feature importances × feature value direction)
    clf = model_pipeline.named_steps['clf']
    importances = pd.Series(clf.feature_importances_,
                             index=metadata['feature_columns'])
    top_drivers = [
        {
            'feature': feat,
            'direction': 'positive' if X_row[feat].values[0] >
                df_features[feat].median() else 'negative'
        }
        for feat in importances.nlargest(3).index
    ]

    return {
        'resident_id':    resident_id,
        'readiness_score': round(score, 4),
        'readiness_tier':  tier,
        'flag':            score >= threshold,
        'top_drivers':     top_drivers,
        'model_version':   metadata['model_version'],
        'threshold_used':  threshold,
        'disclaimer':      ('Score is a decision-support tool. '
                            'Final reintegration decisions require '
                            'qualified social worker assessment.')
    }

with open('model_artifacts/feature_metadata.json') as f:
    meta = json.load(f)

df_indexed = df.reset_index(drop=True)

# Demo: score resident #5
demo_result = predict_readiness(5, df_indexed, gb_pipeline, meta)
print('Demo prediction for resident_id=5:')
print(json.dumps(demo_result, indent=2))

In [ ]:
# ── .NET C# Integration Snippet ───────────────────────────────────────────────
# The following shows how the .NET API controller calls the Python
# inference microservice. This code runs in the .NET backend, NOT in Python.

dotnet_snippet = '''
// ReintegrationReadinessController.cs  (.NET 10 / C#)
// ─────────────────────────────────────────────────────

[ApiController]
[Route("api/residents/{residentId}/readiness-score")]
[Authorize(Roles = "Admin,Staff")]  // Staff-only: never exposed to donors
public class ReintegrationReadinessController : ControllerBase
{
    private readonly IHttpClientFactory _httpClientFactory;
    private readonly IConfiguration     _config;

    public ReintegrationReadinessController(
        IHttpClientFactory httpClientFactory, IConfiguration config)
    {
        _httpClientFactory = httpClientFactory;
        _config            = config;
    }

    [HttpGet]
    public async Task<IActionResult> GetReadinessScore(int residentId)
    {
        // 1. Build feature payload from SQL (feature engineering done in DB layer)
        var features = await _featureService.GetResidentFeaturesAsync(residentId);
        if (features == null)
            return NotFound($"Resident {residentId} not found.");

        // 2. Call Python inference microservice
        var mlServiceUrl = _config["MLService:BaseUrl"];  // stored in env/secrets
        var client = _httpClientFactory.CreateClient();
        var payload = new { resident_id = residentId, features = features };
        var response = await client.PostAsJsonAsync(
            $"{mlServiceUrl}/predict/reintegration-readiness", payload);

        if (!response.IsSuccessStatusCode)
            return StatusCode(502, "ML service unavailable.");

        var result = await response.Content
                          .ReadFromJsonAsync<ReadinessScoreResponse>();

        // 3. Log score to database for audit trail
        await _auditService.LogReadinessScoreAsync(residentId, result, User.Identity.Name);

        return Ok(result);
    }
}
'''

print(dotnet_snippet)

In [ ]:
# ── Pipeline Summary ──────────────────────────────────────────────────────────
print('=' * 65)
print(' REINTEGRATION READINESS PIPELINE — SUMMARY')
print('=' * 65)
print(f'  Dataset:            {len(df)} residents, {len(feature_cols)} features')
print(f'  Target:             reintegration_status == Completed')
print(f'  Positive rate:      {y.mean():.1%} ({y.sum()} of {len(y)} residents)')
print()
print('  Model Comparison (5-Fold Stratified CV):')
print(f'    Logistic Regression:  AUC = {lr_auc.mean():.3f} ± {lr_auc.std():.3f}')
print(f'    Random Forest:        AUC = {rf_auc.mean():.3f} ± {rf_auc.std():.3f}')
print(f'    Gradient Boosting:    AUC = {gb_auc.mean():.3f} ± {gb_auc.std():.3f}  ← DEPLOYED')
print()
print(f'  Deployment threshold:  {DEPLOYMENT_THRESHOLD}')
print(f'  Production model:      model_artifacts/reintegration_readiness_v1.joblib')
print(f'  Explanatory model:     model_artifacts/reintegration_explanatory_v1.joblib')
print()
print('  Readiness tier breakdown (current residents):')
for tier, count in result['readiness_tier'].value_counts().items():
    print(f'    {tier:<25s}: {count} residents')
print()
print('  KEY LIMITATION: n=60 is small for supervised ML.')
print('  Retrain when n >= 150 for production-grade confidence.')
print('  All scores displayed with disclaimer requiring SW sign-off.')
print('=' * 65)